# From hdWGCNA (R) to pySCENIC (Python): gene regulatory networks in the aging mouse heart

This notebook runs a full **pySCENIC** (Single-Cell rEgulatory Network Inference and Clustering)
analysis on the same single-cell dataset used in your `WGCNA_24_hdwgcna_HVG.Rmd` pipeline, and
reuses the **same logic** -- per-cell-type analysis, correlation with age, hub/significant feature
tables -- so that WGCNA modules and SCENIC regulons can be cross-validated against each other.

Based on:
- Speed comparison R vs. Python (why we use GRNBoost2/arboreto instead of GENIE3-in-R for step 1):
  <https://github.com/aertslab/SCENICprotocol/blob/master/notebooks/Figure%20-%20Speed%20comparison%20R%20and%20Python.ipynb>
- Exploring SCENIC output (regulons, AUCell, RSS, binarization -- adapted here to Python/pySCENIC):
  <http://htmlpreview.github.io/?https://github.com/aertslab/SCENIC/blob/master/Tutorials_JupyterNotebooks/SCENIC_tutorial_2-ExploringOutput.html>

**Concept mapping between your WGCNA pipeline and SCENIC:**

| hdWGCNA concept | SCENIC equivalent |
|---|---|
| co-expression module | regulon (a TF + its regulated target genes) |
| hub gene (top kME) | regulon transcription factor (TF) + its high-confidence targets |
| kME (module membership) | regulon AUC (per-cell regulon activity score) |
| module eigengene | mean regulon AUC per mouse (a "regulon eigengene") |
| module-trait (age) correlation | regulon-age correlation (same Pearson + FDR logic) |
| `gene_celltype` pseudobulk feature | regulon computed within one `final_celltype` |

**Data reused from your pipeline:** this notebook reads the same `heart_for_liana.h5ad` you already
exported for the LIANA+ notebook (raw counts in `.layers['counts']`, lognorm in `.X`, and
`final_celltype` / `sample_id` / `age` in `.obs`), plus `hub_genes_by_module.csv`,
`module_age_correlation.csv` and `all_genes_by_module.csv` from hdWGCNA.

**Species note:** mouse data (Tabula Muris Senis, heart) -> we use the **mouse (mgi)** cisTarget
databases, motif annotation table, and mouse TF list throughout, not the human ones.


## 0. R-side step: nothing new needed

If you already ran the export chunk from the LIANA+ notebook, `heart_for_liana.h5ad` already has
everything SCENIC needs: raw counts, cell type, mouse ID, and age. No additional R step is required.

If you have **not** run that export yet, use the chunk from the previous notebook (Section 0 there)
before continuing here.


In [1]:
import os
os.makedirs("resources", exist_ok=True)
%cd resources

C:\Users\olaia\Desktop\WGCNA_results\SCENIC\resources


C:\Users\olaia\anaconda3\envs\pyscenic_env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import sys
import urllib.request

os.makedirs("resources", exist_ok=True)

urls = {
    "mm_mgi_tfs.txt": "https://raw.githubusercontent.com/aertslab/pySCENIC/master/resources/mm_mgi_tfs.txt",
    "mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather": "https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather",
    "mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather": "https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather",
    "motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl": "https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl"
}

def descargar_en_bloques(url, destino):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response:
        size_total = int(response.info().get('Content-Length', 0))
        block_size = 1024 * 1024  # Escribe de 1 MB en 1 MB para no saturar memoria
        descargado = 0
        
        with open(destino, 'wb') as file:
            while True:
                buffer = response.read(block_size)
                if not buffer:
                    break
                file.write(buffer)
                descargado += len(buffer)
                if size_total > 0:
                    porcentaje = int(descargado * 100 / size_total)
                    mb_down = descargado / (1024 * 1024)
                    mb_tot = size_total / (1024 * 1024)
                    sys.stdout.write(f"\r  └─ Progreso: {mb_down:.1f} MB de {mb_tot:.1f} MB ({porcentaje}%)")
                    sys.stdout.flush()
    print()

for nombre_archivo, url in urls.items():
    destino = os.path.join("resources", nombre_archivo)
    
    # Si el archivo ya se descargó correctamente (más de 100 KB), lo omite
    if os.path.exists(destino) and os.path.getsize(destino) > 100000:
        mb_actuales = os.path.getsize(destino) / (1024 * 1024)
        print(f"✓ {nombre_archivo} ya existe ({mb_actuales:.1f} MB). Omitiendo.")
        continue
        
    print(f"Descargando {nombre_archivo}...")
    try:
        descargar_en_bloques(url, destino)
        print(f"  ✓ COMPLETADO: {nombre_archivo}\n")
    except Exception as e:
        print(f"\n   Error en {nombre_archivo}: {e}\n")

print(" Todos los archivos listos en la carpeta 'resources/'.")

Descargando mm_mgi_tfs.txt...
  └─ Progreso: 0.0 MB de 0.0 MB (100%)
  ✓ COMPLETADO: mm_mgi_tfs.txt

✓ mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather ya existe (227.3 MB). Omitiendo.
✓ mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather ya existe (226.2 MB). Omitiendo.
✓ motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl ya existe (107.9 MB). Omitiendo.
 Todos los archivos listos en la carpeta 'resources/'.


## 2. Imports

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import pickle
from pathlib import Path

import numpy as np

# Patch removed NumPy aliases for older library support
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'unicode_'):
    np.unicode_ = np.str_
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

# Proceed with remaining imports
import pandas as pd
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

from dask.diagnostics import ProgressBar

from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell
from pyscenic.export import add_scenic_metadata
from pyscenic.binarization import binarize
from pyscenic.rss import regulon_specificity_scores
from pyscenic.plotting import plot_binarization, plot_rss

%matplotlib inline
sns.set_style('whitegrid')


In [4]:
# Si el directorio actual termina en 'resources', sube un nivel a 'SCENIC'
if Path.cwd().name == "resources":
    os.chdir("..")

# Ahora Path(".") apuntará correctamente a SCENIC
BASE_DIR = Path(".").resolve()

RESOURCES_DIR = BASE_DIR / "resources"
RESULTS_DIR   = BASE_DIR / "scenic_results"
DATA_DIR= BASE_DIR / "input_data"

# Crear la carpeta de resultados en SCENIC
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Comprobación de rutas
print("Directorio de trabajo corregido:", BASE_DIR)
print("Carpeta Resources:              ", RESOURCES_DIR)
print("Carpeta Results:                ", RESULTS_DIR)


# Rutas a archivos de entrada (resources)
TFS_FNAME               = RESOURCES_DIR / "mm_mgi_tfs.txt"
MOTIF_ANNOTATIONS_FNAME = RESOURCES_DIR / "motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl"
DATABASE_GLOB           = str(RESOURCES_DIR / "mm10_*full_tx_v10_clust.genes_vs_motifs.rankings.feather")

# Rutas a archivos de salida (scenic_results)
ADJACENCIES_FNAME = RESULTS_DIR / "adjacencies.tsv"
MOTIFS_FNAME      = RESULTS_DIR / "motifs.csv"
REGULONS_FNAME    = RESULTS_DIR / "regulons.p"
AUC_MTX_FNAME     = RESULTS_DIR / "auc_mtx.csv"

Directorio de trabajo corregido: C:\Users\olaia\Desktop\WGCNA_results\SCENIC
Carpeta Resources:               C:\Users\olaia\Desktop\WGCNA_results\SCENIC\resources
Carpeta Results:                 C:\Users\olaia\Desktop\WGCNA_results\SCENIC\scenic_results


## 3. Load data and prepare the expression matrix

pySCENIC's GRN step expects a cells x genes matrix (raw or lightly-filtered counts work best for
GRNBoost2). We reuse the same `final_celltype` / `sample_id` / `age` annotation as the WGCNA and
LIANA+ notebooks, and apply the same kind of gene filtering already used to build the WGCNA HVG set
(expressed in a minimum fraction of cells).


In [5]:
adata = sc.read_h5ad(DATA_DIR / "heart_for_liana.h5ad")

hub_genes     = pd.read_csv(DATA_DIR / "hub_genes_by_module.csv")     # gene_celltype, module, celltype, gene, kME
module_age    = pd.read_csv(DATA_DIR / "module_age_correlation.csv")  # module, correlation, p_value, n_mice, fdr
all_genes_mod = pd.read_csv(DATA_DIR / "all_genes_by_module.csv")     # gene_celltype, module, celltype, gene(, kME)

sig_modules = module_age.loc[module_age['p_value'] < 0.05, 'module'].tolist()
print("Age-associated modules (p_value<0.05):", sig_modules)

adata


Age-associated modules (p_value<0.05): ['midnightblue', 'lightcyan', 'orange', 'yellow', 'tan', 'brown', 'darkorange']


AnnData object with n_obs × n_vars = 10330 × 20633
    obs: 'final_celltype', 'sample_id', 'age', 'tech'
    layers: 'counts'

In [6]:
# Chronological age order, same approach as the WGCNA / LIANA+ notebooks
age_order = sorted(adata.obs['age'].unique(),
                    key=lambda a: float(''.join(c for c in a if c.isdigit() or c == '.')))
adata.obs['age'] = pd.Categorical(adata.obs['age'], categories=age_order, ordered=True)
adata.obs['age_numeric'] = adata.obs['age'].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)

groupby    = 'final_celltype'
sample_key = 'sample_id'

print(adata.obs['final_celltype'].value_counts())


final_celltype
Fibroblasts          4041
EndothelialCells     3423
Leukocytes           1286
EndocardialCells      708
Myocytes              454
SmoothMuscleCells     418
Name: count, dtype: int64


In [7]:
# Gene filtering: keep genes expressed in at least 1% of cells (consistent in spirit with the
# HVG / expression filters already applied upstream in your Rmd before WGCNA)
sc.pp.filter_genes(adata, min_cells=int(0.01 * adata.n_obs))
print(f"Genes kept for SCENIC: {adata.n_vars}")

# Expression matrix for GRNBoost2: raw counts, cells x genes, as a plain DataFrame
ex_matrix = pd.DataFrame(
    adata.layers['counts'].toarray() if not isinstance(adata.layers['counts'], np.ndarray) else adata.layers['counts'],
    index=adata.obs_names, columns=adata.var_names,
)
ex_matrix.shape


Genes kept for SCENIC: 13314


(10330, 13314)

## 4. Step 1 -- Co-expression modules with GRNBoost2 (`arboreto`)

This is the step the *Speed comparison R and Python* notebook benchmarks: GRNBoost2 (Python,
`arboreto`) is dramatically faster than GENIE3 (R) for this stage, which is why the SCENIC
protocol recommends running GRN inference in Python even when the rest of the analysis (e.g. your
WGCNA) stays in R.


In [8]:
tf_names = load_tf_names(str(TFS_FNAME))
print(f"Mouse TFs in the reference list: {len(tf_names)}")
print(f"Of which present in your filtered expression matrix: {len(set(tf_names) & set(ex_matrix.columns))}")


Mouse TFs in the reference list: 1721
Of which present in your filtered expression matrix: 1101


!pip uninstall dask-expr -y
!pip install "dask==2024.2.1" "distributed==2024.2.1" --force-reinstall

In [12]:
from dask.distributed import Client, LocalCluster

# 1. Crear un cluster local usando hilos en lugar de procesos independientes
cluster = LocalCluster(processes=False)
client = Client(cluster)

# 2. Ejecutar grnboost2 pasando el cliente explícitamente
adjacencies = grnboost2(
    expression_data=ex_matrix,
    tf_names=tf_names,
    client_or_address=client,
    verbose=True
)

# 3. Guardar y mostrar resultados
adjacencies.to_csv(ADJACENCIES_FNAME, sep='\t', index=False)
adjacencies.sort_values('importance', ascending=False).head(10)


preparing dask client
parsing input
creating dask graph
1 partitions
computing dask graph
not shutting down client, client was created externally
finished


,TF,target,importance
735,Tcf21,Bgn,46.966266
734,Tcf15,Cav1,41.933602
734,Tcf15,Ly6c1,41.647307
735,Tcf21,Dpt,39.920907
734,Tcf15,Cd36,39.901524
580,Rab7,Fth1,39.482193
735,Tcf21,Gsn,38.435337
735,Tcf21,Mgp,38.226055
735,Tcf21,Dcn,37.353805
580,Rab7,Hsp90ab1,36.631791


## 5. Step 2 -- Regulon prediction with cisTarget (motif pruning)

Co-expression modules from step 1 still contain many indirect/false-positive TF-target links.
cisTarget prunes each module down to targets whose promoters/enhancers are enriched for the TF's
binding motif, using the two mouse ranking databases downloaded above (promoter-proximal + distal).


In [9]:
if ADJACENCIES_FNAME.exists():
    print(f"[checkpoint] Cargando adjacencies ya calculadas desde: {ADJACENCIES_FNAME}")
    adjacencies = pd.read_csv(ADJACENCIES_FNAME, sep='\t')

[checkpoint] Cargando adjacencies ya calculadas desde: C:\Users\olaia\Desktop\WGCNA_results\SCENIC\scenic_results\adjacencies.tsv


In [10]:
db_fnames = glob.glob(DATABASE_GLOB)
def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]

dbs = [RankingDatabase(fname=fn, name=name(fn)) for fn in db_fnames]
print("Ranking databases loaded:")
for db in dbs:
    print(" -", db.name)


Ranking databases loaded:
 - mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings
 - mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings


In [11]:
modules = list(modules_from_adjacencies(adjacencies, ex_matrix))
print(f"Co-expression modules before pruning: {len(modules)}")



2026-08-18 11:46:26,966 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-08-18 11:46:27,293 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].

2026-08-18 11:46:54,687 - pyscenic.utils - INFO - Creating modules.


Co-expression modules before pruning: 3965


In [14]:
import numpy as np
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float

In [15]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(processes=False)
client = Client(cluster)

with ProgressBar():
    motifs_df = prune2df(
        dbs, modules, str(MOTIF_ANNOTATIONS_FNAME),
        client_or_address=client
    )

motifs_df.to_csv(MOTIFS_FNAME)
regulons = df2regulons(motifs_df)

with open(REGULONS_FNAME, 'wb') as f:
    pickle.dump(regulons, f)

print(f"Regulons after cisTarget pruning: {len(regulons)}")
regulon_sizes = pd.Series({r.name: len(r) for r in regulons}).sort_values(ascending=False)
regulon_sizes.head(15)

2026-08-18 12:34:46,359 - distributed.worker - WARNING - Compute Failed
Key:       ('modules2df-to_pyarrow_string-4ca43909e623086fd3a059a296da2687', 984)
Function:  execute_task
args:      ((subgraph_callable-2e244412951ec1150f59b6836175582e, (functools.partial(<function modules2df at 0x00000208A72855A0>, module2features_func=functools.partial(<function module2features_auc1st_impl at 0x00000208A7285480>, rank_threshold=1500, auc_threshold=0.05, nes_threshold=3.0, filter_for_annotation=True), weighted_recovery=False), FeatherRankingDatabase(name="mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings"), [Regulon(name='Regulon for Cebpa', gene2weight=frozendict.frozendict({'C1qb': 8.78370889461834, 'Tmem176b': 5.23259797289662, 'Tyrobp': 4.288816665397189, 'Cd207': 4.02906433574199, 'Lyz2': 3.915019110667453, 'Csf1r': 3.910058845311116, 'Fcgr3': 3.5042415568577145, 'C1qa': 3.274432695965536, 'Slc7a8': 3.185936134502168, 'Rac3': 3.1843502883474524, 'Rgs1': 3.023921826673001, 

KeyError: 'Field "1110008P14Rik" does not exist in schema'

In [ ]:
# Regulon size distribution
plt.figure(figsize=(6, 4))
sns.histplot(regulon_sizes, bins=30, color='#4575b4')
plt.xlabel('n target genes per regulon')
plt.ylabel('n regulons')
plt.title('Regulon size distribution')
plt.tight_layout()
plt.show()


## 6. Step 3 -- Regulon activity per cell (AUCell)

For each cell, AUCell scores how enriched each regulon's target genes are among the cell's
top-expressed genes. This is the SCENIC analog of a per-cell "module eigengene".


In [ ]:
auc_mtx = aucell(ex_matrix, regulons, num_workers=4)
auc_mtx.to_csv(AUC_MTX_FNAME)
auc_mtx.iloc[:5, :5]


In [ ]:
# Global look at regulon activity: hierarchical clustering of cells x regulons
sns.clustermap(
    auc_mtx.sample(min(500, auc_mtx.shape[0]), random_state=0),  # subsample cells for a readable plot
    cmap='mako', figsize=(10, 8), xticklabels=False, yticklabels=False,
    row_cluster=True, col_cluster=True,
)
plt.suptitle('Regulon activity (AUCell) -- clustered cells x regulons (subsample)', y=1.02)
plt.show()


## 7. Exploring the output (adapted from `SCENIC_tutorial_2-ExploringOutput`)

Attach the AUCell matrix back onto `adata`, build a regulon-activity-based embedding, look at
regulon on/off binarization, and rank regulons by cell-type specificity (RSS) -- the direct SCENIC
analog of asking "which module is specific to which cell type" in hdWGCNA.


In [ ]:
add_scenic_metadata(adata, auc_mtx, regulons)

# Regulon columns are named "Regulon(TF_name)" -- convenient regex-based selection
regulon_cols = [c for c in adata.obs.columns if c.startswith('Regulon(')]
print(f"{len(regulon_cols)} regulon activity columns added to adata.obs")


In [ ]:
# UMAP based purely on regulon activity (instead of gene expression) -- this is the classic
# SCENIC "regulon-based cell states" view
sc.pp.neighbors(adata, use_rep='X_aucell') if 'X_aucell' in adata.obsm else None

if 'X_aucell' not in adata.obsm:
    adata.obsm['X_aucell'] = auc_mtx.reindex(adata.obs_names).values
    sc.pp.neighbors(adata, use_rep='X_aucell')

sc.tl.umap(adata)
sc.pl.umap(adata, color=['final_celltype', 'age'], ncols=2, frameon=False,
           title=['Regulon-activity UMAP -- cell type', 'Regulon-activity UMAP -- age'])


In [ ]:
# Binarize the AUCell matrix (bimodal on/off calling per regulon) -- same idea as your
# WGCNA hub-gene thresholding, but derived from the AUC distribution itself
bin_mtx, thresholds = binarize(auc_mtx, num_workers=4)
bin_mtx.to_csv(RESULTS_DIR / "auc_binarized.csv")

# Example: inspect the binarization for the single most cell-type-variable regulon
example_regulon = auc_mtx.var().sort_values(ascending=False).index[0]
fig, ax = plt.subplots(figsize=(6, 4))
plot_binarization(auc_mtx, example_regulon, thresholds[example_regulon], ax=ax)
ax.set_title(f'AUC binarization -- {example_regulon}')
plt.tight_layout()
plt.show()


In [ ]:
# Regulon Specificity Score (RSS) per final_celltype -- which regulons best mark each cell type
rss = regulon_specificity_scores(auc_mtx, adata.obs['final_celltype'])
rss.to_csv(RESULTS_DIR / "rss_by_celltype.csv")
rss.head()


In [ ]:
# RSS ranking plot per cell type (top regulons that define each cell type's identity)
fig, axes = plt.subplots(1, len(rss.index), figsize=(4 * len(rss.index), 4), sharey=True)
for ax, celltype in zip(axes, rss.index):
    plot_rss(rss, celltype, top_n=8, ax=ax)
    ax.set_title(celltype)
plt.tight_layout()
plt.show()


In [ ]:
# Cell-type specificity heatmap: top regulons per cell type by RSS (complements the ranking plots)
top_per_celltype = pd.concat([rss.loc[ct].sort_values(ascending=False).head(8) for ct in rss.index])
top_regulons = top_per_celltype.index.unique()

plt.figure(figsize=(6, max(4, 0.3 * len(top_regulons))))
sns.heatmap(rss[top_regulons].T if set(top_regulons).issubset(rss.columns) else rss.loc[:, top_regulons].T,
            cmap='rocket_r', cbar_kws={'label': 'RSS'})
plt.title('Regulon Specificity Score (top regulons per cell type)')
plt.tight_layout()
plt.show()


## 8. Bridge to your WGCNA logic: regulon-age correlation

Exactly like your `module_trait_table` in the Rmd, we build a mouse-level "regulon eigengene"
(mean AUCell score per mouse, optionally per cell type), correlate it with age, and apply the same
FDR<0.05 threshold you used for `module_age_correlation.csv`.


In [ ]:
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

def regulon_age_correlation(auc_mtx, obs, groupby_col=None, sample_col='sample_id', age_col='age_numeric'):
    """Mirrors your R module-trait correlation: mean regulon AUC per mouse (optionally per cell type),
    correlated against age across mice."""
    df = auc_mtx.copy()
    df[sample_col] = obs[sample_col].values
    df[age_col] = obs[age_col].values
    if groupby_col is not None:
        df[groupby_col] = obs[groupby_col].values
        group_iter = df.groupby(groupby_col)
    else:
        group_iter = [(None, df)]

    records = []
    for group_val, sub in group_iter:
        mouse_means = sub.groupby(sample_col).agg(
            {**{c: 'mean' for c in auc_mtx.columns}, age_col: 'mean'}
        )
        for regulon in auc_mtx.columns:
            if mouse_means[regulon].std() == 0:
                continue
            r, p = pearsonr(mouse_means[regulon], mouse_means[age_col])
            records.append({
                'celltype': group_val, 'regulon': regulon,
                'correlation': r, 'p_value': p, 'n_mice': mouse_means.shape[0],
            })
    out = pd.DataFrame(records)
    out['fdr'] = multipletests(out['p_value'], method='fdr_bh')[1]
    return out.sort_values('fdr')

# Global (all cell types pooled), matching the overall module_age_correlation table
regulon_age_global = regulon_age_correlation(auc_mtx, adata.obs, groupby_col=None)

# Per cell type, matching the gene_celltype granularity of your WGCNA pseudobulk
regulon_age_by_celltype = regulon_age_correlation(auc_mtx, adata.obs, groupby_col='final_celltype')

regulon_age_by_celltype.to_csv(RESULTS_DIR / "regulon_age_correlation_by_celltype.csv", index=False)
regulon_age_global.to_csv(RESULTS_DIR / "regulon_age_correlation_global.csv", index=False)

sig_regulons = regulon_age_by_celltype[regulon_age_by_celltype['fdr'] < 0.05]
print(f"Significant age-associated regulons (FDR<0.05, per cell type): {len(sig_regulons)}")
sig_regulons.sort_values('fdr').head(15)


In [ ]:
# Barplot of regulon-age correlations, same style as the module-age correlation barplot
m = regulon_age_by_celltype.copy()
m = m.sort_values('correlation')
m['significant'] = m['fdr'] < 0.05
m['label'] = m['regulon'] + ' (' + m['celltype'].astype(str) + ')'

top_show = pd.concat([m[m['significant']].head(15), m[m['significant']].tail(15)]).drop_duplicates()
if top_show.empty:
    top_show = pd.concat([m.head(15), m.tail(15)])

plt.figure(figsize=(6, max(4, 0.3 * len(top_show))))
colors = top_show['significant'].map({True: '#d73027', False: '#bdbdbd'})
plt.barh(top_show['label'], top_show['correlation'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Pearson correlation with age')
plt.title('Regulon-age correlation per cell type (red = FDR < 0.05)')
plt.tight_layout()
plt.show()


## 9. Cross-validation against your hdWGCNA modules

Two complementary checks:

1. **TF overlap** -- are the transcription factors of age-associated regulons also hub genes of
   age-associated WGCNA modules (in the same cell type)?
2. **Target-gene enrichment** -- are a regulon's target genes, as a set, enriched for genes from a
   given WGCNA module (hypergeometric test), per cell type?


In [ ]:
# 9.1 Direct TF overlap
age_hub_genes = hub_genes[hub_genes['module'].isin(sig_modules)][['gene', 'celltype', 'module']]

sig_regulons_annot = sig_regulons.copy()
sig_regulons_annot['tf'] = sig_regulons_annot['regulon'].str.replace(r'\(\+\)|\(-\)', '', regex=True).str.strip()

overlap = sig_regulons_annot.merge(
    age_hub_genes, left_on=['tf', 'celltype'], right_on=['gene', 'celltype'], how='inner'
)
print(f"Age-associated regulons whose TF is also an age-associated hdWGCNA hub gene: {len(overlap)}")
overlap[['regulon', 'celltype', 'module', 'correlation', 'fdr']]


In [ ]:
# 9.2 Target-gene set enrichment: regulon targets vs. WGCNA module genes (per cell type), hypergeometric test
from scipy.stats import hypergeom

regulon_targets = {r.name: set(r.genes) for r in regulons}

def enrich_regulon_vs_module(regulon_name, celltype, module_name, universe):
    targets = regulon_targets.get(regulon_name, set())
    module_genes = set(all_genes_mod.loc[
        (all_genes_mod['celltype'] == celltype) & (all_genes_mod['module'] == module_name), 'gene'
    ])
    k = len(targets & universe)
    K = len(module_genes & universe)
    N = len(universe)
    n = len(targets & universe)
    overlap_genes = targets & module_genes
    if N == 0 or n == 0 or K == 0:
        return None
    pval = hypergeom.sf(len(overlap_genes) - 1, N, K, n)
    return {
        'regulon': regulon_name, 'celltype': celltype, 'module': module_name,
        'n_targets': len(targets), 'n_module_genes': len(module_genes),
        'n_overlap': len(overlap_genes), 'p_value': pval,
        'overlap_genes': sorted(overlap_genes),
    }

universe = set(adata.var_names)
enrichment_records = []
for _, row in sig_regulons_annot.drop_duplicates(['regulon', 'celltype']).iterrows():
    for module_name in sig_modules:
        res = enrich_regulon_vs_module(row['regulon'], row['celltype'], module_name, universe)
        if res is not None:
            enrichment_records.append(res)

regulon_module_enrichment = pd.DataFrame(enrichment_records)
if not regulon_module_enrichment.empty:
    regulon_module_enrichment['fdr'] = multipletests(regulon_module_enrichment['p_value'], method='fdr_bh')[1]
    regulon_module_enrichment = regulon_module_enrichment.sort_values('fdr')

regulon_module_enrichment.to_csv(RESULTS_DIR / "regulon_vs_wgcna_module_enrichment.csv", index=False)
regulon_module_enrichment.head(15)


In [ ]:
# Heatmap: -log10(FDR) of regulon-target vs. module-gene enrichment, regulons x modules
if not regulon_module_enrichment.empty:
    heat = regulon_module_enrichment.pivot_table(
        index='regulon', columns='module', values='fdr', aggfunc='min'
    )
    heat = -np.log10(heat.fillna(1))

    plt.figure(figsize=(6, max(4, 0.3 * heat.shape[0])))
    sns.heatmap(heat, cmap='rocket_r', cbar_kws={'label': '-log10(FDR)'})
    plt.title('Regulon targets vs. WGCNA age-modules: enrichment significance')
    plt.tight_layout()
    plt.show()
else:
    print("No significant regulon-module pairs to plot yet -- check n_targets/n_module_genes above.")


## 10. Optional: tie together WGCNA + SCENIC + LIANA+

If you also ran the LIANA+ notebook, `liana_res_age_module_relevant.csv` lists LR interactions
linked to age-associated WGCNA modules. Here we check whether the ligands/receptors in that table
are themselves targets of an age-associated regulon -- connecting **who is transcriptionally
reprogrammed with age** (SCENIC) to **who is signaling differently with age** (LIANA+), through the
same WGCNA modules that anchor both analyses.


In [ ]:
liana_path = DATA_DIR / "liana_res_age_module_relevant.csv"
if liana_path.exists():
    liana_age_relevant = pd.read_csv(liana_path)
    lr_genes = set(liana_age_relevant['ligand_complex']) | set(liana_age_relevant['receptor_complex'])

    age_regulon_targets = set()
    for regulon_name in sig_regulons_annot['regulon'].unique():
        age_regulon_targets |= regulon_targets.get(regulon_name, set())

    triple_overlap = lr_genes & age_regulon_targets
    print(f"LR genes (from age-relevant LIANA+ interactions) that are also targets of an "
          f"age-associated regulon: {len(triple_overlap)}")
    print(sorted(triple_overlap))
else:
    print("liana_res_age_module_relevant.csv not found -- run the LIANA+ notebook first to enable this step.")


## 11. Summary of generated outputs

- `scenic_results/adjacencies.tsv` -- raw GRNBoost2 TF-target co-expression edges.
- `scenic_results/motifs.csv`, `regulons.p` -- cisTarget-pruned regulons (motifs table + pickled regulon objects).
- `scenic_results/auc_mtx.csv`, `auc_binarized.csv` -- per-cell regulon activity (continuous and binarized).
- `scenic_results/rss_by_celltype.csv` -- Regulon Specificity Score per cell type.
- `scenic_results/regulon_age_correlation_by_celltype.csv` / `_global.csv` -- regulon-age correlations, the SCENIC analog of `module_age_correlation.csv`.
- `scenic_results/regulon_vs_wgcna_module_enrichment.csv` -- hypergeometric enrichment of regulon targets in WGCNA age-modules.

**Suggested next steps:**
1. Re-run cisTarget pruning (Section 5) restricted to specific cell types by subsetting `ex_matrix`/`modules` beforehand, if you want fully separate GRNs per `final_celltype` (closer still to your per-celltype WGCNA pseudobulk design, at higher compute cost).
2. For large datasets, move the GRNBoost2 step (Section 4) to the `pyscenic grn` command-line tool on a multi-core machine or cluster, then resume this notebook from Section 5 using the saved `adjacencies.tsv`.
3. Cross-check `regulon_vs_wgcna_module_enrichment.csv` against `liana_res_age_module_relevant.csv` for biological stories that appear in all three layers (WGCNA + SCENIC + LIANA+).
